# Lab 08: Dynamic Programming

**Author:** Bo Tang

## Setup

See the [README](README.md#getting-started) for installation. Skip the `%pip` cell if `gurobipy` and `numpy` are already installed in this kernel.

In [ ]:
# Optional: run only if these packages are missing from this kernel.
%pip install gurobipy numpy

In [1]:
import gurobipy as gp
import numpy as np
from gurobipy import GRB

## Dynamic Programming

In the reliability problem below, choosing units for one component spends money that could improve another. A locally best choice need not give the best complete system. Dynamic programming compares each choice together with the best continuation it leaves available.

At stage $n$, the state $s_n$ must contain enough information to determine feasible actions, their returns, and the next state without the full decision history. An action $x_n$ earns $h_n(s_n,x_n)$ and moves to $g_n(s_n,x_n)$. Define $f_n(s_n)$ as the best total return from stage $n$ onward. For an additive reward objective,

$$
f_n(s_n)=\max_{x_n\text{ feasible}}\left[h_n(s_n,x_n)+f_{n+1}(g_n(s_n,x_n))\right].
$$

For each action, combine its immediate return with the best remaining return; then choose the largest total. Set the terminal value and work backward, storing the maximizing actions. The reliability example uses a product of returns instead of a sum.

Recursion expresses these dependencies. Memoization avoids solving the same state repeatedly; the Fibonacci example isolates that computational idea before we apply it to optimization.

## Recursion and Memoization

A recursive function calls itself on a smaller problem and stops at a base case. Memoization stores solved subproblems so repeated calls can reuse their answers.

### Fibonacci Example

With $F(0)=0$ and $F(1)=1$, the recurrence is

$$
F(n)=F(n-1)+F(n-2)\qquad(n\geq2).
$$

In [2]:
def fibonacci(n):
    # Base cases
    if n == 0:
        return 0
    elif n == 1:
        return 1
    # Recursive case
    return fibonacci(n-1) + fibonacci(n-2)

In [3]:
import time
start = time.time()
print(fibonacci(40))
end = time.time()
print(f"{end - start:.7f} sec.")

102334155
7.5871799 sec.


The recursion repeats subproblems, giving an exponential number of calls (bounded above by $O(2^n)$). For example, `F(3)` is computed twice in this call tree:

```
                      F(5)
                    /      \
              F(4)           F(3)
             /    \          /   \
         F(3)    F(2)      F(2)  F(1)
        /  \     /  \      /  \
     F(2) F(1) F(1) F(0) F(1) F(0)
    /  \
F(1) F(0)
```

Cache each computed value in a dictionary:

In [4]:
def fibonacci(n, memo={}):
    # Base cases
    if n == 0:
        return 0
    elif n == 1:
        return 1
    # Check if value is already computed
    if n in memo:
        return memo[n]
    # Recursive calculation with memoization
    memo[n] = fibonacci(n-1, memo) + fibonacci(n-2, memo)
    return memo[n]

In [5]:
import time
start = time.time()
print(fibonacci(40))
end = time.time()
print(f"{end - start:.7f} sec.")

102334155
0.0000091 sec.


**Result:** Both implementations return $F(40)=102334155$. Memoization avoids recomputing the same Fibonacci values. Timings depend on the machine; redefine the memoized function before timing it again, since its default dictionary retains cached values.

## System Reliability

A system works only if all four components work. Choose one, two, or three parallel units for each component using the reliability and cost tables below.

### Probability of Functioning

| # of Units | Reliability of A | Reliability of B | Reliability of C | Reliability of D |
|------------|------------------|------------------|------------------|------------------|
| 1          | 0.5              | 0.6              | 0.7              | 0.5              |
| 2          | 0.6              | 0.7              | 0.8              | 0.7              |
| 3          | 0.8              | 0.8              | 0.9              | 0.9              |

### Cost

| # of Units | Cost of A | Cost of B | Cost of C | Cost of D |
|------------|-----------|-----------|-----------|-----------|
| 1          | \$100     | \$200     | \$100     | \$200     |
| 2          | \$200     | \$400     | \$300     | \$300     |
| 3          | \$300     | \$500     | \$400     | \$400     |

### Goal

Assuming the four component subsystems function independently, and given a budget of \$1,000, determine how the money should be allocated to maximize the system's reliability.

In [6]:
# Parameters
reliabilities = {
    "A": [0, 0.5, 0.6, 0.8],
    "B": [0, 0.6, 0.7, 0.8],
    "C": [0, 0.7, 0.8, 0.9],
    "D": [0, 0.5, 0.7, 0.9]
}
costs = {
    "A": [0, 100, 200, 300],
    "B": [0, 200, 400, 500],
    "C": [0, 100, 300, 400],
    "D": [0, 200, 300, 400]
}

# Indices
components = ["A", "B", "C", "D"]
units = [1, 2, 3]

### Exercise: Integer Programming Formulation

Use binary variables to select exactly one unit count per component. Enforce the \$1,000 budget and maximize the sum of log reliabilities.

Report the chosen unit counts, total cost, and product of the four selected reliabilities. The printed reliability must be the product, not the log-objective value.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
# Create a model
m = gp.Model("System Reliability")
# Variables
x = m.addVars(components, units, vtype=GRB.BINARY, name="x")

# TODO: Set the log-reliability objective, add selection and budget constraints, and solve.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
print("Solutions:")
obj_val = 1
for c, u in x:
    if x[c,u].x >= 1 - 1e-3:
        print("Component {}: {} units.".format(c, u))
        obj_val *= reliabilities[c][u]
print("Maximum Reliability: {:.4f}".format(obj_val))

### Exercise: Dynamic Programming Formulation

Let $f(i,b)$ be the best reliability for components $i,\ldots,N-1$ with budget $b$ remaining before choosing units for component $i$. Choosing $u\in\{1,2,3\}$ units costs $c_i(u)$ and contributes reliability $r_i(u)$:

$$
f(i,b)=\max_{u:c_i(u)\leq b}\left[r_i(u)f(i+1,b-c_i(u))\right].
$$

Use 1 when no components remain and 0 for an infeasible remaining allocation. Start at $f(0,1000)$ and store both the best reliability and the chosen unit counts.

**Discussion questions:**

- Why is the terminal value 1 rather than 0 for a product of reliabilities?
- Which information must be included in a memoization key?

Complete the recursive function and report the allocation, cost, and reliability. Compare with the integer model. Also test a budget of 600, which permits only one unit per component, and a budget below 600, for which no complete allocation is feasible. Use a fresh memo dictionary for each independent test.

In [ ]:
# Recursive dynamic programming function
def dp(comp_ind, budget, memo={}):
    # Base case: no components left
    if comp_ind >= len(components):
        max_reliability, best_units = 1, []
        return max_reliability, best_units
    # Check if solution already computed in memo
    if (comp_ind, budget) in memo:
        return memo[comp_ind, budget]
    # Init setting
    max_reliability = 0
    best_units = []
    c = components[comp_ind]
    # Recursive case: try all unit options for the current component
    for u in units:
        cost = costs[c][u]
        # Check if feasible
        if budget >= cost:
            # TODO: recursive dp
            reliability, units_used = dp()
            # TODO: calculate current reliability
            current_reliability =
            # Get best one
            if current_reliability > max_reliability:
                max_reliability = current_reliability
                best_units = [u] + units_used
    # Memoize
    # TODO: update memory
    memo[comp_ind, budget] =
    # Return the result
    return max_reliability, best_units

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
# Solve
max_reliability, unit_configuration = dp(comp_ind=0, budget=1000)
# Solution
print("Solutions:")
obj_val = 1
for i, c in enumerate(components):
    print("Component {}: {} units.".format(c, unit_configuration[i]))
print("Maximum Reliability: {:.4f}".format(max_reliability))